# Notebook 02 — Preprocesamiento
**Imputación · Split · Scaling · SMOTE**

In [ ]:
!pip install pandas numpy scikit-learn imbalanced-learn matplotlib -q
print('✓ Listo')

In [ ]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

from google.colab import files
uploaded = files.upload()
import io
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']
print(f'✓ Dataset cargado: {df.shape}')

In [ ]:
# Paso 1 — Imputar ceros imposibles con mediana por clase
print('VALORES CERO ANTES DE IMPUTACIÓN:')
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    print(f'  {col}: {(df[col]==0).sum()} ceros')

df_proc = df.copy()
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    df_proc.loc[df_proc[col]==0, col] = np.nan

for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    for cls in [0, 1]:
        median_val = df_proc.loc[df_proc['Outcome']==cls, col].median()
        mask = (df_proc['Outcome']==cls) & (df_proc[col].isna())
        df_proc.loc[mask, col] = median_val

print(f'\n✓ Imputación completada')
print(f'  Nulos restantes: {df_proc[features].isna().sum().sum()}')
print(f'  Mediana glucosa clase 0: {df_proc[df_proc["Outcome"]==0]["Glucose"].median():.1f}')
print(f'  Mediana glucosa clase 1: {df_proc[df_proc["Outcome"]==1]["Glucose"].median():.1f}')

In [ ]:
# Paso 2 — Split estratificado 80/20
X = df_proc[features].values
y = df_proc['Outcome'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'✓ Split completado')
print(f'  Train: {X_train.shape[0]} ({y_train.mean()*100:.1f}% positivos)')
print(f'  Test:  {X_test.shape[0]} ({y_test.mean()*100:.1f}% positivos)')

In [ ]:
# Paso 3 — StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'✓ Scaling aplicado')
print(f'  Media features (train): {X_train_sc.mean(axis=0).round(3)}')
print(f'  Std features  (train): {X_train_sc.std(axis=0).round(3)}')

In [ ]:
# Paso 4 — SMOTE para balancear clases
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
print(f'✓ SMOTE aplicado')
print(f'  Antes:   Clase 0={sum(y_train==0)}, Clase 1={sum(y_train==1)}')
print(f'  Después: Clase 0={sum(y_train_sm==0)}, Clase 1={sum(y_train_sm==1)}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_data, titulo in zip(axes,
        [y_train, y_train_sm],
        ['Antes de SMOTE', 'Después de SMOTE']):
    counts = pd.Series(y_data).value_counts().sort_index()
    ax.bar(['Sin diabetes','Con diabetes'], counts.values,
           color=['#457B9D','#E63946'], alpha=0.85, width=0.5)
    for i,(val) in enumerate(counts.values):
        ax.text(i, val+2, str(val), ha='center', fontweight='bold')
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylim(0, max(counts.values)*1.2)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Efecto de SMOTE en el balance de clases', fontweight='bold')
plt.tight_layout()
plt.savefig('smote_balance.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ Gráfico SMOTE guardado')

In [ ]:
# Guardar dataset procesado
df_proc.to_csv('diabetes_procesado.csv', index=False, encoding='utf-8-sig')
print('✓ Dataset procesado guardado')
import numpy as np
np.save('X_train_sm.npy', X_train_sm)
np.save('y_train_sm.npy', y_train_sm)
np.save('X_test_sc.npy', X_test_sc)
np.save('y_test.npy', y_test)

from google.colab import files
files.download('diabetes_procesado.csv')
print('✓ Descargado — guárdalo en data/processed/')